# SEAWRD

Planetary interior modelling is a computationally demanding exercise, involving a lot of hydrodynamical considerations dependent on the composition and physical properties of an exoplanet. This takes a number of minutes, which grows to be incredibly large when performing hundreds of thousands of simulations.

A cheap approximation is availabe in the form of surrogate models. A neural network can act as a general function learner, i.e., something that maps inputs to outputs, and so we can use pre-ran expensive simulation data to train a small neural network to reproduce the simulation's results with great accuracy in a fraction of time. The hope is to train widely enough for this model to be used on new, never-seen exoplanet data and avoid expensive simulations.

This is what **S**urrogate **E**mulator for **A**quatic **W**orld **R**adius **D**etermination is for! Based on user-provided hyperparameters and data, it can train an appropriate surrogate model to be used in future research as an approximation for the full hydrodynamical simulations, namely as a predictor of the size of the planet.

## Imports

In [1]:
import pandas as pd

from preprocessing_data import DataPreprocessor
from model import DNNManager
from trainer import DNNTrainer

I0000 00:00:1782491701.832547  205551 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782491701.978822  205551 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/ampar/miniconda3/envs/codeastro/lib/python3.10/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
I0000 00:00:1782491703.936859  205551 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different co

## Loading data

We are going to be using the simulation results from [Aguichine et al. (2021)](https://iopscience.iop.org/article/10.3847/1538-4357/abfa99), stored locally. This is from the result of expensive hydrodynamical simulations. The descriptions of each column is as follows:

* **x_core'** - the specific core mass fraction of the planet, equal to x_core / (1-x_H20)
    i.e., x_core' = 0.325 refers to an Earth-like CMF, regardles of the amount of water present
* **x_H20** - the water mass fraction of the planet
* **T_irr** - the irradiation temperature of the planet
* **T_b** - the boundary temperature i.e., at the outer boundary of the core.
* **M_b** - mass of the interior of the planet
* **M_a** - mass of the atmosphere of the planet
* **R_b** - radius of the interior of the planet
* **R_a** - radius of the atmosphere of the planet

In [2]:
# Reading the data from the file
DATA_PATH = "DNN_data_IOP_Aguichine2021.dat"
data = pd.read_table(DATA_PATH, sep=r"\s+")

data.head(5)

,x_core',x_H2O,T_irr,T_b,M_b,M_a,R_b,R_a,errcode
0,0.0,0.0,400.0,400.0,0.200000,0.0,0.663520,0.0,0
1,0.0,0.0,400.0,400.0,0.254855,0.0,0.713469,0.0,0
2,0.0,0.0,400.0,400.0,0.324755,0.0,0.767308,0.0,0
3,0.0,0.0,400.0,400.0,0.413828,0.0,0.821989,0.0,0
4,0.0,0.0,400.0,400.0,0.527330,0.0,0.881669,0.0,0


The `DataPreprocessor` class is designed to parse Pandas dataframes like this into a format appropriate for model training, specifically validating and inferring feature & label columns & names, performing a training/test data split, and also creating a calibrated normaliser for use in the model architecture.

We need to specify what in the dataframe we are classing as features. In this case, we are deciding to simplify the model by combining the different masses and radii into one value for the planet, i.e., $M_p = M_b + M_a,  R_p = R_b + R_a$.

We also determine which column is the quality-control; in this datafile, if `errcode = 1` for any row, it means an error has occured within the simulation. We have chosen the simple method of removing any rows with errors in them.

We can also specify what fraction of the inputted data should be test data (`TEST_SIZE`) and whether to produce a normaliser, which we strongly recommend

In [3]:
# Setting up the data to be processed properly
feature_names = ["x_core'", "x_H2O", "T_irr", "T_b", "M_p"]
label_name = "R_p"

# This is to do with errors in the data; any row with a 1 in the errcode column will not be included
quality_column = "errcode"
quality_threshold = 0

# Options for the DataPreprocessor
TEST_SIZE = 0.2
RANDOM_STATE = 42
NORMALIZE = True
dp = DataPreprocessor(df=data,
                      features=feature_names,
                      label=label_name,
                      test_size=TEST_SIZE,
                      random_state=RANDOM_STATE,
                      quality_column=quality_column,
                      quality_value=quality_threshold,
                      normalize=NORMALIZE)

normaliser, train_features, test_features, train_labels, test_labels = dp.get_training_data()

print(f"Train features shape: {train_features.shape}")
print(train_features.head(5))
print(f"\nTrain labels shape: {train_labels.shape}")
print(train_labels.head(5))
print(f"\nTest features shape: {test_features.shape}")
print(test_features.head(5))
print(f"\nTest labels shape: {test_labels.shape}")
print(test_labels.head(5))

Train features shape: (11846, 5)
       x_core'  x_H2O   T_irr       T_b        M_p
12852      0.8    0.7  1000.0  3620.986  12.323118
11803      0.7    0.7  1300.0  4166.968   2.261279
2662       0.1    1.0  1100.0  4007.148   9.673014
11886      0.8    0.2   400.0  2399.488   1.773041
14798      0.9    0.9  1300.0  4082.193  19.997184

Train labels shape: (11846,)
12852    2.929560
11803    2.729390
2662     3.285913
11886    1.502180
14798    3.489365
Name: R_p, dtype: float64

Test features shape: (2962, 5)
    x_core'  x_H2O  T_irr    T_b       M_p
1       0.0    0.0  400.0  400.0  0.254855
4       0.0    0.0  400.0  400.0  0.527330
5       0.0    0.0  400.0  400.0  0.671964
9       0.0    0.0  400.0  400.0  1.771734
11      0.0    0.0  400.0  400.0  2.876900

Test labels shape: (2962,)
1     0.713469
4     0.881669
5     0.946804
9     1.252312
11    1.432190
Name: R_p, dtype: float64


W0000 00:00:1782491705.470097  205551 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Optional demo: create a DNN Model

The class `DNNManager` is used to create and manage dense neural network (DNN) models. In the normal processing of the code, it is largely in the back-end, with a separate class `DNNTrainer` initiating an instance for training purposes. Here, however, is a little demostration of the `DNNManager` class.

A DNN model has several different layers of neurons. The first layer is an input layer, combined with a normalisation layer, which will take a row of training features (i.e., one value for every feature). There is then a number of hidden layers, followed by an output layer, which will produce a label(s) (in this case, we want to predict the radius of the planet $R_p$)

On first use, we have to decide some important hyperparameters of the model - the number of hidden layers and the number of neurons in each layer. While adding more layers and more neurons is likely to increase model performance (i.e., make it more accurate) up to some point, it will also greatly increase training time. It is up to you to balance the architecture!

In [4]:
# Generate the model
NUM_LAYERS = 4
NUM_NEURONS = 8

dnn_manager = DNNManager.from_new_model(num_layers=NUM_LAYERS,
                                        num_neurons=NUM_NEURONS,
                                        input_shape=train_features.shape[1:],
                                        normaliser=normaliser,
                                        num_outputs=1)
dnn_model = dnn_manager.model

print(dnn_model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ feature_normalizer              │ (None, 5)              │            11 │
│ (Normalization)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 284 (1.11 KB)

 Trainable params: 273 (1.07 KB)

 Non-trainable params: 11 (48.00 B)

None


Every model has a `model_name`, which describes the architecture above.

'R' stands for Rectangular, which is the type of basic network represented here. Then you have `num_layers`x`num_neurons` to show how many hidden layers of how many neurons are included. Then you have `num_inputs`i and `num_outputs`o.

In [5]:
print(dnn_manager.model_name)

R(4x8_5i_1o)


It is also possible to save and load a model into the class, allowing for extra-training and usage of previously-created models. This is available through the `DNNManager.from_previous_model()` constructor and the `save_model_version()` method.

For more informations on Dense Neural Networks, see [here](https://www.scribd.com/document/480813741/AML-03-Dense-Neural-Networks).

## Model Training

A model is initialised with random small weights and biases, and so will be very poorly performing. The `DNNTrainer` class is designed to train a model architecture to be able to predict the planetary radius given a the feature vector.

In the back-end, this creates and uses a `DNNManager` instance. Therefore, depending on whether you are intending to load a previous version or not, the keywords you will need to input into the constructor will change.

We will start from a new model, which therefore needs speifying `NUM_LAYERS` and `NUM_NEURONS`. We can also specify how many epochs (`NUM_EPOCHS`) to train each model for (more on that later!) and the version the final model will be called. Note, we have set `NUM_EPOCHS` to a very small value here in order to demonstrate functionality, you will most definitely want to run it for more epochs.

There are also other hyperparameters:
* **learning_rate** - a very important hyperparameter, essentially how large a change should be made in the model's weights and biases during each training step. This is decreased during training to smoothly approach minima.
* **batch_size** - how many feature vectors to include in a single training step; the model is evaluated based on the mean-squared error of all of these predictions, rather than just a single one.
* **validation_split** - during training, the performance of the model is evaluated by testing its predictions on validation data. Validation data is not used by the model in training, and is split from the inputted training data, based on this fraction. 

In [ ]:
# Model hyperparameters
NUM_LAYERS = 4
NUM_NEURONS = 8
VERSION = 1

# Training hyperparameters
NUM_EPOCHS = 100
LEARNING_RATE = 5e-3
BATCH_SIZE = 128
VALIDATION_SPLIT = 0.2

dnn_trainer = DNNTrainer(load_existing=False,
                        num_layers=NUM_LAYERS,
                        num_neurons=NUM_NEURONS,
                        num_epochs=NUM_EPOCHS,
                        input_shape=train_features.shape[1:],
                        normaliser=normaliser,
                        num_outputs=1,
                        version=VERSION,
                        learning_rate=LEARNING_RATE,
                        batch_size=BATCH_SIZE,
                        validation_split=VALIDATION_SPLIT)

Necause our DNNs are small in size, the final result depends heavily on our initial (random) state.  To combat this, we will train the same model architecture multiple (`NUM_MODELS`) times, each one having different random initial conditions (the `seed`). While training, we will keep track of statistics about each model's performance and save only the best performing model in the end.

During trainer, extra actions are also taken!

In [ ]:
# Train the models
NUM_MODELS = 3
rp_means, rp_stds, losses, val_losses = dnn_trainer.train_models(input_features=train_features,
                                                                input_labels=train_labels,
                                                                test_features=test_features,
                                                                test_labels=test_labels,
                                                                num_models=NUM_MODELS)

Training model 0/3:

Epoch: 0, loss:2.1043,  mean_squared_error:2.1043,  val_loss:0.1513,  val_mean_squared_error:0.1513,  
....................................................................................................
Epoch: 100, loss:0.0012,  mean_squared_error:0.0012,  val_loss:0.0012,  val_mean_squared_error:0.0012,  
..................................................Restoring model weights from the end of the best epoch: 135.
Final val_loss of model 0/3: 0.0007132863975130022
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Training model 1/3:

Epoch: 0, loss:0.0016,  mean_squared_error:0.0016,  val_loss:0.0008,  val_mean_squared_error:0.0008,  
....................
Epoch 20: ReduceLROnPlateau reducing learning rate to 0.0024999999441206455.
........................
Epoch 44: ReduceLROnPlateau reducing learning rate to 0.0012499999720603228.
....................
Epoch 64: ReduceLROnPlateau reducing learning rate to 0.0006249999860301614.
....................
Epoch 84: ReduceLROnPlatea

With the model finished training, we can see some statistics!

In [ ]:
# Print the architecture performance of the best model
dnn_trainer.print_architecture_performance()

We can also visualise the loss curve of the best model from training, compared against the validation loss this is not going to be great for our very short example but you'll be able to clearly see improvements w/ epochs!

In [ ]:
dnn_trainer.print_loss_curve(log_y=False)